# 99 — Session Recovery (run after any Kaggle restart, ~5 min)

Rebuilds the full working state from the HF backup repo: data loaders,
text embeddings, DINOv2+LoRA model, and (if trained) the GeoPrompt model.
No retraining. Requires the `HF_TOKEN` Kaggle Secret.

After this notebook: continue in 02 (training/eval) or 03 (Darmstadt).

In [ ]:
# Bootstrap: locate rg_geoprompt package (Kaggle dataset or local repo)
# Run every session. No GPU required.
import sys
from pathlib import Path
for cand in ["/kaggle/input/rg-geoprompt-src/src", "../src", "src"]:
    if Path(cand, "rg_geoprompt").exists():
        sys.path.insert(0, str(Path(cand).resolve()))
        break
from rg_geoprompt import paths
print(paths.describe())

In [ ]:
# Installs (idempotent, ~1 min). Run every session.
%pip install -q peft open_clip_torch
print("✓ peft + open_clip_torch installed")

In [ ]:
# REQUIRES: Kaggle T4 GPU
# Rebuild everything from local files or the HF backup repo.
import torch
from rg_geoprompt import paths
from rg_geoprompt.datasets import build_potsdam_loaders
from rg_geoprompt.models_dino_lora import load_dinov2_lora
from rg_geoprompt.models_geoprompt import load_geoprompt
from rg_geoprompt.prompts import load_or_encode_text_embeddings
from rg_geoprompt.utils import ensure_checkpoint

train_loader, val_loader = build_potsdam_loaders()
print(f"✓ loaders ({len(train_loader)}/{len(val_loader)} batches)")

# text embeddings: HF copy → local cache → only then CLIP re-encode
if not paths.TEXT_EMBEDDINGS_PT.exists():
    cached = ensure_checkpoint("text_embeddings.pt", required=False)
    if cached and cached != paths.TEXT_EMBEDDINGS_PT:
        import shutil; shutil.copy(cached, paths.TEXT_EMBEDDINGS_PT)
text_embeddings = load_or_encode_text_embeddings()

model = load_dinov2_lora(ensure_checkpoint("dinov2_lora_best.pth"))
print("✓ DINOv2+LoRA loaded (84.6% mIoU)")

geo_ckpt = ensure_checkpoint("geoprompt_best.pth", required=False)
if geo_ckpt:
    geo_model = load_geoprompt(geo_ckpt, text_embeddings)
    print("✓ GeoPrompt loaded")
else:
    print("  GeoPrompt not yet trained — build it in notebook 02")

print("\n✓ Recovery complete.")

In [ ]:
# Optional smoke test: quick mIoU on a few val batches (~1 min)
import itertools
from torch.utils.data import DataLoader
from rg_geoprompt.metrics import compute_miou

few = list(itertools.islice(iter(val_loader), 5))
class _Tiny:                       # minimal loader wrapper for 5 batches
    def __iter__(self): return iter(few)
    def __len__(self): return len(few)
_, miou = compute_miou(model, _Tiny())
print(f"DINOv2+LoRA quick mIoU on 5 val batches: {miou*100:.1f}% "
      "(full-val reference: 84.6%)")